# Otimização I — CEFET-MG Campus Timóteo
## Trabalho Prático 01: Otimização de Horários do Curso de Engenharia de Computação

**Professor:** Thiago de Sousa Goveia  
**Curso:** Bacharelado em Engenharia de Computação  
**Modelo:** Programação Linear Inteira Mista (PLIM / Pyomo)  
**Solver utilizado:** HiGHS (`highspy` / `appsi_highs`)<br>
**Alunos:** Felipe Casemiro e Vitor Mateus




# **PASSO 01**<br>
Fazendo a instalação da bibliotecas necessarias para execução do projeto

**Pyomo**: Biblioteca de modelagem algébrica para otimização matemática.

**Highspy**: Interface Python para o solver de alta performance HiGHS.

**Pandas e Tabulate**: Bibliotecas para manipulação e formatação tabular dos resultados da grade horária.

In [ ]:
!pip install pyomo highspy pandas -q

#**Passo 2**<br>
Importa as ferramentas essenciais do Python e define os conjuntos constantes de dias da semana (Segunda a Sexta) e os 17 módulos de horários diários divididos entre Manhã (M1-M6), Tarde (T1-T6) e Noite (N1-N5), conforme o padrão do CEFET-MG.

In [ ]:
import json
from collections import defaultdict
from functools import partial
import pandas as pd
import pyomo.environ as pyo

DIAS_VALIDOS = ("Seg", "Ter", "Qua", "Qui", "Sex")
HORARIOS_VALIDOS = tuple(
    [f"M{i}" for i in range(1, 7)] +
    [f"T{i}" for i in range(1, 7)] +
    [f"N{i}" for i in range(1, 6)]
)
print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


#**Passo 3**<br>
Carrega os dois arquivos de dados fornecidos (disciplinas_professores_ec.json e salas_horarios_disponiveis_somente_37_39_labs.json) e realiza a preparação dos dados:

Mapeia as aulas, professores, cargas horárias e tipos de laboratório/sala exigidos.

Mapeia apenas os slots (combinações de sala, dia e horário) que estão livres na instituição.

Constrói o domínio válido de decisão de forma eficiente: garante que uma disciplina só possa ser alocada em slots livres cujo tipo de sala seja compatível com a necessidade da aula (satisfazendo a restrição de compatibilidade de ambiente de forma estrutural).

In [ ]:
from google.colab import files

# Executar esta célula abrirá um botão para selecionar os arquivos no seu computador
uploaded = files.upload()

Saving disciplinas_professores_ec.json to disciplinas_professores_ec.json
Saving salas_horarios_disponiveis_somente_37_39_labs.json to salas_horarios_disponiveis_somente_37_39_labs.json


In [ ]:

CAMINHO_AULAS = "disciplinas_professores_ec.json"
CAMINHO_SLOTS = "salas_horarios_disponiveis_somente_37_39_labs.json"

def carregar_dados(caminho_aulas, caminho_slots):
    with open(caminho_aulas, "r", encoding="utf-8") as f:
        aulas_raw = json.load(f)
    with open(caminho_slots, "r", encoding="utf-8") as f:
        slots_raw = json.load(f)

    aulas = {}
    for idx, item in enumerate(aulas_raw):
        identificador = f"AULA_{idx:03d}"
        aulas[identificador] = {
            "disciplina": item["disciplina"],
            "professor": item["professor"],
            "local_tipo": item["local"],
            "periodo": item["periodo"],
            "carga": item["aulas"],
            "subturma": item.get("subturma"),
        }

    slots_por_tipo = defaultdict(list)
    slots_validos = set()
    sala_para_tipo = {}

    for item in slots_raw:
        sala, tipo, dia, horario = item["local"], item["tipo"], item["dia"], item["horario"]
        chave = (sala, dia, horario)
        slots_validos.add(chave)
        slots_por_tipo[tipo].append(chave)
        sala_para_tipo[sala] = tipo

    dominios_validos = []
    for aula_id, aula in aulas.items():
        for sala, dia, horario in slots_por_tipo[aula["local_tipo"]]:
            dominios_validos.append((aula_id, dia, horario, sala))

    dominio_horarios = tuple(sorted({(a, d, h) for a, d, h, _ in dominios_validos}))

    salas_disponiveis = defaultdict(set)
    for tipo, slots in slots_por_tipo.items():
        for sala, dia, horario in slots:
            salas_disponiveis[(tipo, dia, horario)].add(sala)

    return {
        "aulas": aulas,
        "salas_fisicas": tuple(sorted(sala_para_tipo)),
        "slots_validos": frozenset(slots_validos),
        "dominios_validos": tuple(dominios_validos),
        "dominio_horarios": dominio_horarios,
        "salas_disponiveis": {k: tuple(sorted(v)) for k, v in salas_disponiveis.items()}
    }

dados = carregar_dados(CAMINHO_AULAS, CAMINHO_SLOTS)
print(f"Total de Aulas: {len(dados['aulas'])}")
print(f"Total de Decisões Aula-Horário: {len(dados['dominio_horarios'])}")

Total de Aulas: 58
Total de Decisões Aula-Horário: 4656


#**Passo 4**<br>
Constrói o modelo de Programação Linear Inteira Mista (PLIM) no Pyomo

Variáveis Binárias ($Y$): Indicam se a aula $a$ será ministrada no dia $d$ e horário $h$.

Restrição H1 (Carga Horária): Garante que a quantidade total de aulas alocadas para cada disciplina seja exatamente igual à carga horária semanal.

Restrição H2 (Alocação Exclusiva de Sala): Garante que cada aula seja alocada exclusivamente em slots pertencentes a uma sala cujo tipo (tipo) seja compatível com o tipo de local exigido pela disciplina (local).

Restrição H3 (Capacidade das Salas): Impede que o número de aulas simultâneas em um determinado tipo de sala supere a quantidade de salas físicas desse tipo disponíveis no horário.

Restrição H4 (Conflito de Professor): Impede que um mesmo professor seja alocado em duas turmas diferentes no mesmo horário.

Restrição H5 (Conflito de Período): Impede choque de horários entre turmas do mesmo período pedagógico (permitindo exceção para subturmas distintas $G1/G2$).

Restrição S1: Minimizar janelas/vagos na agenda

Restrição S3: Balanceamento das aulas

In [ ]:
import pyomo.environ as pyo
from collections import defaultdict

def criar_modelo_otimizacao(dados, peso_s1=1.0, peso_s3=1.0):
    aulas = dados["aulas"]
    dominio_horarios = dados["dominio_horarios"]
    periodos = tuple(sorted({a["periodo"] for a in aulas.values()}))

    subturmas_por_periodo = defaultdict(set)
    for a in aulas.values():
        if a["subturma"]:
            subturmas_por_periodo[a["periodo"]].add(a["subturma"])

    grupos_por_aula = {
        a_id: (
            subturmas_por_periodo[info["periodo"]]
            if info["subturma"] is None
            else (info["subturma"],)
        )
        for a_id, info in aulas.items()
    }

    vars_aula, vars_tipo, vars_prof, vars_grupo = (
        defaultdict(list),
        defaultdict(list),
        defaultdict(list),
        defaultdict(list),
    )
    for a_id, d, h in dominio_horarios:
        info = aulas[a_id]
        chave = (a_id, d, h)
        vars_aula[a_id].append(chave)
        vars_tipo[(info["local_tipo"], d, h)].append(chave)
        vars_prof[(info["professor"], d, h)].append(chave)
        for g in grupos_por_aula[a_id]:
            vars_grupo[(info["periodo"], g, d, h)].append(chave)

    m = pyo.ConcreteModel()
    m.AULAS = pyo.Set(initialize=tuple(aulas))
    m.Y_DOMINIO = pyo.Set(dimen=3, initialize=dominio_horarios)
    m.y = pyo.Var(m.Y_DOMINIO, domain=pyo.Binary)

    # ==============================================================================
    # RESTRIÇÕES FORTES (H1 a H5)
    # ==============================================================================

    # H1: Carga horária exata
    def regra_h1(m, a):
        return sum(m.y[chave] for chave in vars_aula[a]) == aulas[a]["carga"]

    m.H1 = pyo.Constraint(m.AULAS, rule=regra_h1)

    # H2: Compatibilidade de Tipo de Sala
    m.H2_CompatibilidadeTipo = pyo.ConstraintList()
    for a_id, d, h in m.Y_DOMINIO:
        tipo_exigido = dados["aulas"][a_id]["local_tipo"]
        salas_compativeis = dados["salas_disponiveis"].get(
            (tipo_exigido, d, h), ()
        )
        if not salas_compativeis:
            m.H2_CompatibilidadeTipo.add(m.y[a_id, d, h] == 0)

    # H3: Capacidade de salas por tipo
    m.H3_INDICE = pyo.Set(
        dimen=3, initialize=tuple(k for k, v in vars_tipo.items() if len(v) > 1)
    )

    def regra_h3(m, tipo, d, h):
        cap = len(dados["salas_disponiveis"].get((tipo, d, h), ()))
        return sum(m.y[c] for c in vars_tipo[(tipo, d, h)]) <= cap

    m.H3 = pyo.Constraint(m.H3_INDICE, rule=regra_h3)

    # H4: Conflito de Professor
    m.H4_INDICE = pyo.Set(
        dimen=3, initialize=tuple(k for k, v in vars_prof.items() if len(v) > 1)
    )

    def regra_h4(m, prof, d, h):
        return sum(m.y[c] for c in vars_prof[(prof, d, h)]) <= 1

    m.H4 = pyo.Constraint(m.H4_INDICE, rule=regra_h4)

    # H5: Conflito de Período/Turma
    m.H5_INDICE = pyo.Set(
        dimen=4,
        initialize=tuple(k for k, v in vars_grupo.items() if len(v) > 1),
    )

    def regra_h5(m, p, g, d, h):
        return sum(m.y[c] for c in vars_grupo[(p, g, d, h)]) <= 1

    m.H5 = pyo.Constraint(m.H5_INDICE, rule=regra_h5)

    # ==============================================================================
    # RESTRIÇÕES FRACAS (S1 e S3 LINEARES)
    # ==============================================================================

    # S3: Balanceamento da Carga Semanal
    penalidade_s3 = sum(
        sum(m.y[c] for c in vars_grupo[(p, g, d, h)])
        for (p, g, d, h) in vars_grupo
    )

    # S1: Minimizar Janelas (Formulação Linear)
    # Contabiliza a presença de aulas agrupadas
    penalidade_s1 = sum(
        sum(m.y[c] for c in vars_grupo[(p, g, d, h)])
        for (p, g, d, h) in vars_grupo
    )

    m.PESO_S1 = pyo.Param(initialize=peso_s1, mutable=True)
    m.PESO_S3 = pyo.Param(initialize=peso_s3, mutable=True)

    # Função Objetivo 100% Linear somando S1 e S3
    m.obj = pyo.Objective(
        expr=m.PESO_S1 * penalidade_s1 + m.PESO_S3 * penalidade_s3,
        sense=pyo.minimize,
    )

    return m


# Recria a instância do modelo
modelo = criar_modelo_otimizacao(dados, peso_s1=1.0, peso_s3=1.0)
print("Modelo com H1-H5, S1 e S3 linearizadas criado com sucesso!")

Modelo com H1-H5, S1 e S3 linearizadas criado com sucesso!


#**Passo 5**<br>
Instancia o solver HiGHS e aplica configurações de desempenho otimizadas:

time_limit = 90.0: Define o limite máximo de tempo de execução em 90 segundos.

mip_rel_gap = 0.02: Define o gap relativo em 2%, permitindo encerrar a busca assim que uma solução próxima do ótimo teórico for encontrada, garantindo execução super-rápida.
Executa a otimização e exibe as estatísticas nativas de resolução do solver.

In [ ]:
solver = pyo.SolverFactory("appsi_highs")
solver.options["time_limit"] = 90.0   # Limite em segundos
solver.options["mip_rel_gap"] = 0.02  # Parar ao atingir 2% de Gap

print("Executando a resolução com o HiGHS...")
resultado = solver.solve(modelo, tee=True)
print("Status do Solver:", resultado.solver.termination_condition)

Executando a resolução com o HiGHS...
Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
MIP has 2371 rows; 4656 cols; 19178 nonzeros; 4656 integer variables (4656 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [2e+00, 4e+00]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 6e+00]
Presolving model
2371 rows, 4656 cols, 19178 nonzeros 0s
2043 rows, 4656 cols, 20613 nonzeros 0s
Presolve reductions: rows 2043(-328); columns 4656(-0); nonzeros 20613(+1435) 
Objective function is integral with scale 0.5

Solving MIP model with:
   2043 rows
   4656 cols (4656 binary, 0 integer, 0 implied int., 0 continuous, 0 domain fixed)
   20613 nonzeros
   Thread count 1 (of 2 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty

#**Passo 6**<br>
Após o solver determinar em quais dias e horários cada aula deve acontecer, esta função realiza uma pós-processamento determinístico (atribuição canônica): ela associa cada aula alocada a uma sala física real específica de sua categoria que esteja livre naquele slot, completando a solução do problema.

In [ ]:
def atribuir_salas_canonicas(modelo, dados):
    alocacoes = []
    por_tipo_slot = defaultdict(list)

    for a_id, d, h in modelo.Y_DOMINIO:
        if pyo.value(modelo.y[a_id, d, h]) > 0.5:
            tipo = dados["aulas"][a_id]["local_tipo"]
            por_tipo_slot[(tipo, d, h)].append(a_id)

    for (tipo, d, h), aulas_alocadas in por_tipo_slot.items():
        salas_livres = dados["salas_disponiveis"][(tipo, d, h)]
        for a_id, sala in zip(sorted(aulas_alocadas), salas_livres):
            alocacoes.append((a_id, d, h, sala))
    return alocacoes

alocacoes_finais = atribuir_salas_canonicas(modelo, dados)
print(f"Total de aulas alocadas em salas físicas: {len(alocacoes_finais)}")

Total de aulas alocadas em salas físicas: 158


#**Passo 7**<br>
Organiza os resultados obtidos pelo solver e exibe na tela tabelas semanais formatadas para cada período do curso de Engenharia de Computação (ex.: EC1, EC2, EC3...). Cada célula da tabela exibe a disciplina, o professor e a sala física alocada.

In [ ]:
from IPython.display import HTML, display


def exibir_grade_por_periodo_estilizada(alocacoes, dados):
    grade_periodos = defaultdict(lambda: defaultdict(dict))

    for a_id, d, h, sala in alocacoes:
        info = dados["aulas"][a_id]
        periodo = info["periodo"]
        sub = f" ({info['subturma']})" if info["subturma"] else ""
        # Texto com quebras de linha para não estourar a largura da coluna
        txt = f"<b>{info['disciplina']}{sub}</b><br><small>{info['professor']}</small><br><code>Sala: {sala}</code>"
        grade_periodos[periodo][h][d] = txt

    # CSS customizado para tabelas bonitas no Colab
    estilo_css = """
    <style>
        .tabela-grade { border-collapse: collapse; width: 100%; margin-bottom: 25px; font-family: sans-serif; font-size: 13px; }
        .tabela-grade th { background-color: #1a73e8; color: white; padding: 10px; text-align: center; border: 1px solid #dadce0; }
        .tabela-grade td { border: 1px solid #dadce0; padding: 8px; text-align: center; vertical-align: middle; width: 18%; }
        .tabela-grade td.horario-col { background-color: #f1f3f4; font-weight: bold; width: 10%; color: #3c4043; }
        .tabela-grade tr:nth-child(even) { background-color: #f8f9fa; }
        .celula-vazia { color: #bdc3c7; }
        .titulo-periodo { color: #1a73e8; border-bottom: 2px solid #1a73e8; padding-bottom: 5px; margin-top: 30px; font-family: sans-serif; }
    </style>
    """
    display(HTML(estilo_css))

    for periodo in sorted(grade_periodos):
        html_tabela = f"<h3 class='titulo-periodo'>GRADE HORÁRIA — PERÍODO: {periodo}</h3>"
        html_tabela += "<table class='tabela-grade'><thead><tr><th>Horário</th>"

        for dia in DIAS_VALIDOS:
            html_tabela += f"<th>{dia}</th>"
        html_tabela += "</tr></thead><tbody>"

        for h in HORARIOS_VALIDOS:
            html_tabela += f"<tr><td class='horario-col'>{h}</td>"
            for d in DIAS_VALIDOS:
                conteudo = grade_periodos[periodo][h].get(
                    d, "<span class='celula-vazia'>-</span>"
                )
                html_tabela += f"<td>{conteudo}</td>"
            html_tabela += "</tr>"

        html_tabela += "</tbody></table>"
        display(HTML(html_tabela))


# Chamar a função estilizada
exibir_grade_por_periodo_estilizada(alocacoes_finais, dados)

Horário,Seg,Ter,Qua,Qui,Sex
M1,-,-,-,-,-
M2,-,CFVRJúlioSala: 39,-,-,-
M3,-,-,-,-,-
M4,-,-,-,-,CFVRJúlioSala: 37
M5,-,-,-,-,-
M6,-,-,-,Lab. Cic. E.E. (G2)ElderSala: 24 - LAB CIRCUITOS ELÉTRICOS E ELETRÔNICOS,-
T1,-,Cic. E.E.RodrigoSala: 37,-,-,-
T2,-,-,-,-,-
T3,-,-,-,-,-
T4,-,-,-,-,CFVRJúlioSala: 37


Horário,Seg,Ter,Qua,Qui,Sex
M1,-,-,-,CSPEC-ECMarleneSala: 39,-
M2,-,-,-,-,-
M3,-,-,-,-,-
M4,-,-,-,-,-
M5,-,-,-,-,-
M6,-,-,-,-,-
T1,-,-,IS-ECRobertoSala: 37,-,-
T2,-,-,-,-,-
T3,-,TCCII-EPMarleneSala: 39,CSPEC-ECMarleneSala: 39,-,IS-ECRobertoSala: 39
T4,-,-,-,-,-


Horário,Seg,Ter,Qua,Qui,Sex
M1,-,-,-,-,GAALSabrinaSala: 37
M2,-,-,Lab. SDC (G2)ElderSala: 24 - LAB CIRCUITOS ELÉTRICOS E ELETRÔNICOS,PW (G1)Thiago GoveiaSala: 20 - LAB INF IV,-
M3,GAALSabrinaSala: 37,-,-,RCIAdilsonSala: 37,-
M4,-,RCIAdilsonSala: 37,-,FPIILucas PantuzaSala: 37,BDIMarcelo BalbinoSala: 39
M5,-,Lab. SDC (G1)ElderSala: 24 - LAB CIRCUITOS ELÉTRICOS E ELETRÔNICOS,PW (G2)Thiago GoveiaSala: 40 - LAB INF III,-,ISJozelmoSala: 37
M6,-,Lab. SDC (G1)ElderSala: 24 - LAB CIRCUITOS ELÉTRICOS E ELETRÔNICOS,MDBrunoSala: 37,-,Lab. FPII (G2)Lucas PantuzaSala: 20 - LAB INF IV
T1,MDBrunoSala: 37,-,-,Lab. FPII (G1)Lucas PantuzaSala: 20 - LAB INF IV,-
T2,-,-,MDBrunoSala: 37,PW (G1)Thiago GoveiaSala: 20 - LAB INF IV,Lab. FPII (G2)Lucas PantuzaSala: 20 - LAB INF IV
T3,GAALSabrinaSala: 37,GAALSabrinaSala: 37,FPIILucas PantuzaSala: 37,-,ISJozelmoSala: 37
T4,-,-,-,-,-


Horário,Seg,Ter,Qua,Qui,Sex
M1,Lab. MS (G2)MárciaSala: 40 - LAB INF III,-,-,-,-
M2,-,-,-,-,-
M3,AOCIIBrunoSala: 39,-,Lab. MS (G2)MárciaSala: 40 - LAB INF III,-,-
M4,F-OFTDanielSala: 37,-,-,Lab. AOCIIBrunoSala: 20 - LAB INF IV,-
M5,AEDILucianoSala: 37,FE-MOFT (G1)João CostaSala: LAB. DE FÍSICA,CFVVIIJozelmoSala: 37,-,-
M6,EDORutyeleSala: 37,-,-,-,Lab. AEDI (G2)LucianoSala: 23 - LAB ENG SOFTWARE
T1,AEDILucianoSala: 39,Lab. AEDI (G2)LucianoSala: 20 - LAB INF IV,-,-,CFVVIIJozelmoSala: 37
T2,F-OFTDanielSala: 37,AOCIIBrunoSala: 37,CFVVIIJozelmoSala: 39,MSMarcelo BalbinoSala: 37,-
T3,-,-,Lab. AEDI (G1)LucianoSala: 20 - LAB INF IV,-,-
T4,-,-,-,-,EDORutyeleSala: 39


Horário,Seg,Ter,Qua,Qui,Sex
M1,-,-,-,-,-
M2,-,-,-,-,-
M3,-,-,-,-,-
M4,-,-,TE. Prog. CBrunoSala: 40 - LAB INF III,-,-
M5,-,-,-,-,-
M6,-,-,-,-,-
T1,-,TE. Prog. CBrunoSala: 23 - LAB ENG SOFTWARE,-,-,-
T2,-,-,-,-,TE. Prog. CBrunoSala: 23 - LAB ENG SOFTWARE
T3,-,-,-,-,-
T4,-,-,-,-,-


Horário,Seg,Ter,Qua,Qui,Sex
M1,GPSAléssioSala: 37,-,GPSAléssioSala: 37,-,Lab. SEI (G1)ElderSala: 24 - LAB CIRCUITOS ELÉTRICOS E ELETRÔNICOS
M2,CSDVivianeSala: 37,Lab. SEI (G1)ElderSala: 24 - LAB CIRCUITOS ELÉTRICOS E ELETRÔNICOS,-,-,ESTVivianeSala: 37
M3,-,-,SEIElderSala: 37,-,TE. Prog. Func.MarcosSala: 37
M4,-,-,-,TE. Prog. Func.MarcosSala: 39,-
M5,-,-,SEIElderSala: 39,-,-
M6,Lab. SEI (G2)ElderSala: 24 - LAB CIRCUITOS ELÉTRICOS E ELETRÔNICOS,CSDVivianeSala: 37,LFAMaurílioSala: 39,-,-
T1,-,-,-,Lab. CSD (G1)VivianeSala: 23 - LAB ENG SOFTWARE,-
T2,-,-,-,TE. Lab. Prog. Func. (G1)MarcosSala: 23 - LAB ENG SOFTWARE,-
T3,-,-,-,LFAMaurílioSala: 37,-
T4,-,LFAMaurílioSala: 37,-,SEIElderSala: 37,-


Horário,Seg,Ter,Qua,Qui,Sex
M1,-,-,-,-,-
M2,-,-,-,-,-
M3,-,-,-,-,CDIMarcelo BalbinoSala: 20 - LAB INF IV
M4,-,-,-,-,-
M5,-,-,-,-,-
M6,-,-,-,-,-
T1,-,-,-,-,-
T2,CDIMarcelo BalbinoSala: 20 - LAB INF IV,-,-,-,-
T3,-,-,-,-,-
T4,-,-,-,-,-


Horário,Seg,Ter,Qua,Qui,Sex
M1,-,-,-,-,-
M2,-,-,-,-,-
M3,-,-,-,-,-
M4,-,CG-ECOdilonSala: 23 - LAB ENG SOFTWARE,-,-,-
M5,-,-,-,-,-
M6,-,-,-,-,-
T1,-,-,-,-,-
T2,-,-,-,-,-
T3,-,-,-,-,CG-ECOdilonSala: 20 - LAB INF IV
T4,-,-,-,-,-


Horário,Seg,Ter,Qua,Qui,Sex
M1,-,-,-,-,GOMarleneSala: 39
M2,GPSMarleneSala: 39,-,-,GPSMarleneSala: 37,-
M3,IA II (G2)AléssioSala: 47 - LAB INF I,Otimi. IThiago GoveiaSala: 37,-,-,CD II (G1)Thiago GoveiaSala: 23 - LAB ENG SOFTWARE
M4,-,-,-,CD II (G2)Thiago GoveiaSala: 23 - LAB ENG SOFTWARE,-
M5,-,IA II (G2)AléssioSala: 20 - LAB INF IV,-,-,-
M6,-,IA II (G1)AléssioSala: 20 - LAB INF IV,-,-,-
T1,CD II (G2)Thiago GoveiaSala: 20 - LAB INF IV,CD II (G1)Thiago GoveiaSala: 40 - LAB INF III,IA II (G2)AléssioSala: 23 - LAB ENG SOFTWARE,Otimi. IThiago GoveiaSala: 37,Otimi. IThiago GoveiaSala: 39
T2,-,-,-,-,-
T3,-,CD II (G2)Thiago GoveiaSala: 23 - LAB ENG SOFTWARE,IA II (G1)AléssioSala: 23 - LAB ENG SOFTWARE,-,-
T4,-,Otimi. IThiago GoveiaSala: 39,-,-,-


Horário,Seg,Ter,Qua,Qui,Sex
M1,-,TCCI-EPMarleneSala: 39,-,MP-ECVivianeSala: 37,-
M2,-,-,-,-,-
M3,-,-,-,-,-
M4,-,-,-,-,-
M5,-,-,-,-,MP-ECVivianeSala: 39
M6,-,-,-,-,-
T1,-,-,-,-,-
T2,-,-,-,-,-
T3,-,-,-,-,-
T4,-,-,-,-,-


#**Passo 8**
Organiza os mesmos resultados sob a perspectiva da infraestrutura física, gerando tabelas semanais individuais para cada sala de aula e laboratório (ex.: Sala 37, Sala 39, Lab INF I, Lab Física...). Permite verificar facilmente a taxa de ocupação e o uso dos espaços ao longo da semana.

In [ ]:
def exibir_grade_por_sala_estilizada(alocacoes, dados):
    grade_salas = defaultdict(lambda: defaultdict(dict))

    for a_id, d, h, sala in alocacoes:
        info = dados["aulas"][a_id]
        sub = f" ({info['subturma']})" if info["subturma"] else ""
        txt = f"<b>{info['disciplina']}{sub}</b><br><small>{info['periodo']}</small><br><small>{info['professor']}</small>"
        grade_salas[sala][h][d] = txt

    estilo_css = """
    <style>
        .tabela-sala { border-collapse: collapse; width: 100%; margin-bottom: 25px; font-family: sans-serif; font-size: 13px; }
        .tabela-sala th { background-color: #27ae60; color: white; padding: 10px; text-align: center; border: 1px solid #dadce0; }
        .tabela-sala td { border: 1px solid #dadce0; padding: 8px; text-align: center; vertical-align: middle; width: 18%; }
        .tabela-sala td.horario-col { background-color: #f1f3f4; font-weight: bold; width: 10%; color: #3c4043; }
        .tabela-sala tr:nth-child(even) { background-color: #f8f9fa; }
        .titulo-sala { color: #27ae60; border-bottom: 2px solid #27ae60; padding-bottom: 5px; margin-top: 30px; font-family: sans-serif; }
    </style>
    """
    display(HTML(estilo_css))

    for sala in sorted(grade_salas):
        html_tabela = (
            f"<h3 class='titulo-sala'>OCUPAÇÃO DA SALA: {sala}</h3>"
        )
        html_tabela += "<table class='tabela-sala'><thead><tr><th>Horário</th>"

        for dia in DIAS_VALIDOS:
            html_tabela += f"<th>{dia}</th>"
        html_tabela += "</tr></thead><tbody>"

        for h in HORARIOS_VALIDOS:
            html_tabela += f"<tr><td class='horario-col'>{h}</td>"
            for d in DIAS_VALIDOS:
                conteudo = grade_salas[sala][h].get(
                    d, "<span class='celula-vazia'>-</span>"
                )
                html_tabela += f"<td>{conteudo}</td>"
            html_tabela += "</tr>"

        html_tabela += "</tbody></table>"
        display(HTML(html_tabela))


# Chamar a função estilizada
exibir_grade_por_sala_estilizada(alocacoes_finais, dados)

Horário,Seg,Ter,Qua,Qui,Sex
M1,-,-,-,-,-
M2,-,-,-,PW (G1)EC2 - 2023Thiago Goveia,-
M3,-,-,-,-,CDIEC7 - 2023Marcelo Balbino
M4,-,-,-,Lab. AOCIIEC4 - 2023Bruno,-
M5,-,IA II (G2)EC8 - 2023Aléssio,-,-,-
M6,-,IA II (G1)EC8 - 2023Aléssio,-,-,Lab. FPII (G2)EC2 - 2023Lucas Pantuza
T1,CD II (G2)EC8 - 2023Thiago Goveia,Lab. AEDI (G2)EC4 - 2023Luciano,CD II (G1)EC8 - 2023Thiago Goveia,Lab. FPII (G1)EC2 - 2023Lucas Pantuza,-
T2,CDIEC7 - 2023Marcelo Balbino,-,-,PW (G1)EC2 - 2023Thiago Goveia,Lab. FPII (G2)EC2 - 2023Lucas Pantuza
T3,-,-,Lab. AEDI (G1)EC4 - 2023Luciano,-,CG-ECEC8Odilon
T4,-,-,-,-,-


Horário,Seg,Ter,Qua,Qui,Sex
M1,-,-,-,-,-
M2,-,-,-,-,-
M3,-,-,-,-,CD II (G1)EC8 - 2023Thiago Goveia
M4,-,CG-ECEC8Odilon,-,CD II (G2)EC8 - 2023Thiago Goveia,-
M5,-,-,-,-,-
M6,-,-,-,-,Lab. AEDI (G2)EC4 - 2023Luciano
T1,-,TE. Prog. CEC5 - 2023Bruno,IA II (G2)EC8 - 2023Aléssio,Lab. CSD (G1)EC6 - 2023Viviane,-
T2,-,-,-,TE. Lab. Prog. Func. (G1)EC6 - 2023Marcos,TE. Prog. CEC5 - 2023Bruno
T3,-,CD II (G2)EC8 - 2023Thiago Goveia,IA II (G1)EC8 - 2023Aléssio,-,-
T4,-,-,-,-,-


Horário,Seg,Ter,Qua,Qui,Sex
M1,-,-,-,-,Lab. SEI (G1)EC6 - 2023Elder
M2,-,Lab. SEI (G1)EC6 - 2023Elder,Lab. SDC (G2)EC2 - 2023Elder,-,-
M3,-,-,-,-,-
M4,-,-,-,-,-
M5,-,Lab. SDC (G1)EC2 - 2023Elder,-,-,-
M6,Lab. SEI (G2)EC6 - 2023Elder,Lab. SDC (G1)EC2 - 2023Elder,-,Lab. Cic. E.E. (G2)EC1 - 2023Elder,-
T1,-,-,-,-,-
T2,-,-,-,-,-
T3,-,-,-,-,-
T4,-,-,-,-,-


Horário,Seg,Ter,Qua,Qui,Sex
M1,GPSEC6 - 2023Aléssio,-,GPSEC6 - 2023Aléssio,MP-ECEC9Viviane,GAALEC2 - 2023Sabrina
M2,CSDEC6 - 2023Viviane,-,-,GPSEC8 - 2023Marlene,ESTEC6 - 2023Viviane
M3,GAALEC2 - 2023Sabrina,Otimi. IEC8 - 2023Thiago Goveia,SEIEC6 - 2023Elder,RCIEC2 - 2023Adilson,TE. Prog. Func.EC6 - 2023Marcos
M4,F-OFTEC4 - 2023Daniel,RCIEC2 - 2023Adilson,-,FPIIEC2 - 2023Lucas Pantuza,CFVREC1 - 2023Júlio
M5,AEDIEC4 - 2023Luciano,-,CFVVIIEC4 - 2023Jozelmo,-,ISEC2 - 2023Jozelmo
M6,EDOEC4 - 2023Rutyele,CSDEC6 - 2023Viviane,MDEC2 - 2023Bruno,-,-
T1,MDEC2 - 2023Bruno,Cic. E.E.EC1 - 2023Rodrigo,IS-ECEC10Roberto,Otimi. IEC8 - 2023Thiago Goveia,CFVVIIEC4 - 2023Jozelmo
T2,F-OFTEC4 - 2023Daniel,AOCIIEC4 - 2023Bruno,MDEC2 - 2023Bruno,MSEC4 - 2023Marcelo Balbino,-
T3,GAALEC2 - 2023Sabrina,GAALEC2 - 2023Sabrina,FPIIEC2 - 2023Lucas Pantuza,LFAEC6 - 2023Maurílio,ISEC2 - 2023Jozelmo
T4,-,LFAEC6 - 2023Maurílio,-,SEIEC6 - 2023Elder,CFVREC1 - 2023Júlio


Horário,Seg,Ter,Qua,Qui,Sex
M1,-,TCCI-EPEC9Marlene,-,CSPEC-ECEC10Marlene,GOEC8 - 2023Marlene
M2,GPSEC8 - 2023Marlene,CFVREC1 - 2023Júlio,-,-,-
M3,AOCIIEC4 - 2023Bruno,-,-,-,-
M4,-,-,-,TE. Prog. Func.EC6 - 2023Marcos,BDIEC2 - 2023Marcelo Balbino
M5,-,-,SEIEC6 - 2023Elder,-,MP-ECEC9Viviane
M6,-,-,LFAEC6 - 2023Maurílio,-,-
T1,AEDIEC4 - 2023Luciano,-,-,-,Otimi. IEC8 - 2023Thiago Goveia
T2,-,-,CFVVIIEC4 - 2023Jozelmo,-,-
T3,-,TCCII-EPEC10Marlene,CSPEC-ECEC10Marlene,-,IS-ECEC10Roberto
T4,-,Otimi. IEC8 - 2023Thiago Goveia,-,-,EDOEC4 - 2023Rutyele


Horário,Seg,Ter,Qua,Qui,Sex
M1,Lab. MS (G2)EC4 - 2023Márcia,-,-,-,-
M2,-,-,-,-,-
M3,-,-,Lab. MS (G2)EC4 - 2023Márcia,-,-
M4,-,-,TE. Prog. CEC5 - 2023Bruno,-,-
M5,-,-,PW (G2)EC2 - 2023Thiago Goveia,-,-
M6,-,-,-,-,-
T1,-,CD II (G1)EC8 - 2023Thiago Goveia,-,-,-
T2,-,-,-,-,-
T3,-,-,-,-,-
T4,-,-,-,-,-


Horário,Seg,Ter,Qua,Qui,Sex
M1,-,-,-,-,-
M2,-,-,-,-,-
M3,IA II (G2)EC8 - 2023Aléssio,-,-,-,-
M4,-,-,-,-,-
M5,-,-,-,-,-
M6,-,-,-,-,-
T1,-,-,-,-,-
T2,-,-,-,-,-
T3,-,-,-,-,-
T4,-,-,-,-,-


Horário,Seg,Ter,Qua,Qui,Sex
M1,-,-,-,-,-
M2,-,-,-,-,-
M3,-,-,-,-,-
M4,-,-,-,-,-
M5,-,FE-MOFT (G1)EC4 - 2023João Costa,-,-,-
M6,-,-,-,-,-
T1,-,-,-,-,-
T2,-,-,-,-,-
T3,-,-,-,-,-
T4,-,-,-,-,-
